# NB04 V3_1 — Full satellite transfer experiment

Uses the **frozen M2 warm-start full-mosaic maps** and **previously downloaded V3 satellite features**. No Earth Engine calls or new downloads. Native UAV probabilities are copied to the Colab SSD; all aggregations and feature alignment occur locally. Per-site outputs, samples and held-out predictions are cached on Drive.

**Targets**: `mean_probability` = average of valid UAV pixel probabilities (no classification threshold); `cover_fraction_p05` = share of valid UAV pixels with p≥0.5. These are distinct targets. We report both. **Primary validation**: leave-one-site-out (LOSO), 4 forests. Scores quantify transfer to UAV-derived labels, not ground-truth validation of satellite maps.


## 00 — USER SETTINGS

In [ ]:
# ==================== USER SETTINGS ====================
SCALES_M = [10,30,50,75,100,125]
FEATURE_SETS = ['ANNUAL_S2','EMBEDDING','S2_PLUS_EMBEDDING']
MODELS = ['RIDGE','RF','EXTRA_TREES','HIST_GB']
TARGETS = ['mean_probability','cover_fraction_p05']
MAX_SAMPLES_PER_MOSAIC = 8000
MIN_UAV_COVERAGE = .90
SEED = 20261022
COPY_UAV_TO_SSD = True
COPY_FEATURES_TO_SSD = True
FORCE_REBUILD = False  # keep False: resumes cached aggregate/sample/fold files
N_JOBS = 2
SAVE_FIGURES = True
# ========================================================

# BLOCK 06 — targeted high-cover recovery (independent of full Block 04 rerun)
B06_RUN = True
B06_SCALES_M = [75,100,125]
B06_FEATURE_SETS = ['EMBEDDING','S2_PLUS_EMBEDDING']
B06_TARGETS = ['mean_probability','cover_fraction_p05']
B06_WEIGHT_STRENGTHS = [0,2,5]  # 0=standard boosting; >0 emphasizes cover above 20%
B06_HIGH_THRESHOLD = .40
B06_REBUILD = False


## 01 — Mount Drive, audit cached inputs and stage them on local SSD

In [ ]:
%pip -q install rasterio geopandas pandas pyarrow scikit-learn matplotlib scipy
import os, json, math, time, hashlib, shutil, gc, warnings
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from affine import Affine
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OLD=ROOT/'Outputs/V3/04_satellite_Musanga/features'
MAPS=ROOT/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
OUT=ROOT/'Outputs/V3_1/04_satellite_full_warmstart'
CACHE=OUT/'cache';AGG=CACHE/'aggregates';SAMPLE=CACHE/'samples';FOLDS=OUT/'folds';FIG=OUT/'figures'
for p in (CACHE,AGG,SAMPLE,FOLDS,FIG):p.mkdir(parents=True,exist_ok=True)
SSD=Path('/content/musanga_nb04_ssd');(SSD/'uav').mkdir(parents=True,exist_ok=True);(SSD/'features').mkdir(exist_ok=True)
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
SITE_MOSAICS={'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],'Yoko':['Yoko_kvo_georef.tif']}
FAMILY_NAMES={'ANNUAL_S2':['RAW_ANNUAL'],'EMBEDDING':['EMB_ANNUAL'],
              'S2_PLUS_EMBEDDING':['RAW_ANNUAL','EMB_ANNUAL']}
def uavfile(name):return MAPS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
def featfile(site,fam):return OLD/fam/f'{site}_{fam}_{YEARS[site]}_10m.tif'
def staged(path,folder,copy):
    assert path.is_file(),('Missing required cached file',path)
    target=folder/path.name
    if not copy:return path
    srcstat=path.stat()
    if not target.exists() or target.stat().st_size!=srcstat.st_size:
        print('COPY to SSD:',path.name,round(srcstat.st_size/1e9,2),'GB',flush=True)
        tmp=target.with_name(target.name+'.partial')
        shutil.copyfile(path,tmp);os.replace(tmp,target)
    return target
UAV={name:staged(uavfile(name),SSD/'uav',COPY_UAV_TO_SSD) for ns in SITE_MOSAICS.values() for name in ns}
FAMILY={'RAW_ANNUAL','EMB_ANNUAL'}
assert all(featfile(site,fam).is_file() for site in SITE_MOSAICS for fam in FAMILY)
FEATURES={(site,fam):staged(featfile(site,fam),SSD/'features',COPY_FEATURES_TO_SSD)
          for site in SITE_MOSAICS for fam in sorted(FAMILY)}
print('Input audit: ',len(UAV),'UAV maps,',len(FEATURES),'cached predictor stacks')
print('SSD free GB:',round(shutil.disk_usage(SSD).free/1e9,2),flush=True)


## 02 — Aggregate *entire* native probability mosaics to 10–125 m

Area-weighted aggregation onto the **same year-matched Sentinel-2 10 m grid origin**, scaled to 10, 30, 50, 75, 100, 125 m. Intermediate rasters are float32 `[mean_probability, cover_fraction_p05, valid_UAV_coverage]`. No overlapping-mosaic double counting: mosaic cells remain separately indexed. Computing at 75/125 m uses a valid satellite-grid transform and area resampling, not naive array reshaping. Each mosaic/scale is cached on Drive.


In [ ]:
def grid_for(site,scale):
    with rasterio.open(FEATURES[(site,'RAW_ANNUAL')]) as ds:
        assert abs(ds.transform.a-10)<1e-5 and abs(ds.transform.e+10)<1e-5
        f=scale/10.
        return dict(height=math.ceil(ds.height/f),width=math.ceil(ds.width/f),
                    transform=ds.transform*Affine.scale(f,f),crs=ds.crs)
def profile_for(g,count):
    return dict(driver='GTiff',height=g['height'],width=g['width'],count=count,
                crs=g['crs'],transform=g['transform'],dtype='float32',nodata=-9999.,
                tiled=True,blockxsize=256,blockysize=256,compress='deflate',predictor=3,bigtiff='YES')
def warp(src,g,rs=Resampling.average,srcnodata=None):
    result=np.full((g['height'],g['width']),-9999.,np.float32)
    reproject(source=src,destination=result,src_nodata=srcnodata,
              dst_transform=g['transform'],dst_crs=g['crs'],dst_nodata=-9999.,
              resampling=rs,num_threads=2)
    return result
def aggregate_one(site,name,scale):
    dst=AGG/f'{Path(name).stem}_{scale}m.tif'
    if dst.exists() and not FORCE_REBUILD:return dst
    g=grid_for(site,scale)
    with rasterio.open(UAV[name]) as src:
        assert src.count==1 and src.nodata==65535
        # Reproject native probabilities / native hard classes separately; construct
        # binary and valid auxiliary rasters in SSD with block streaming.
        scratch=SSD/f'{Path(name).stem}_native_derived.tif'
        pr=src.profile.copy()
        for key in ('photometric','alpha','extrasamples','NBITS','colorinterp'):pr.pop(key,None)
        pr.update(driver='GTiff',count=3,dtype='float32',nodata=-9999.,
                  tiled=True,blockxsize=256,blockysize=256,compress='deflate',predictor=3)
        with rasterio.open(scratch,'w',**pr) as aux:
            for _,win in src.block_windows(1):
                a=src.read(1,window=win);ok=a!=65535
                prob=np.where(ok,a.astype('float32')/10000.,-9999.).astype('float32')
                hard=np.where(ok,(a>=5000).astype('float32'),-9999.).astype('float32')
                valid=ok.astype('float32')
                aux.write(prob,1,window=win);aux.write(hard,2,window=win);aux.write(valid,3,window=win)
        with rasterio.open(scratch) as aux:
            vals=[warp(rasterio.band(aux,i),g,srcnodata=(-9999. if i<3 else None)) for i in (1,2,3)]
        scratch.unlink(missing_ok=True)
    tmp=dst.with_suffix('.partial.tif')
    with rasterio.open(tmp,'w',**profile_for(g,3)) as ds:
        for i,a in enumerate(vals,1):ds.write(a,i)
        ds.set_band_description(1,'mean_probability')
        ds.set_band_description(2,'cover_fraction_p05')
        ds.set_band_description(3,'uav_valid_coverage')
        ds.update_tags(PRODUCTION_MODEL='M2_warmstart_epoch02',AGGREGATION_M=str(scale),
                       PROB_THRESHOLD='0.5',SOURCE_MOSAIC=name,COMPLETE='YES')
    os.replace(tmp,dst)
    return dst
AGG_LOOKUP={}
for scale in SCALES_M:
    t=time.perf_counter()
    for site,names in SITE_MOSAICS.items():
        for name in names:
            AGG_LOOKUP[(name,scale)]=aggregate_one(site,name,scale)
            print('AGG READY',scale,'m',site,Path(name).stem,'elapsed',round(time.perf_counter()-t,1),'s',flush=True)
print('All requested scale rasters cached.')


## 03 — Build consistent satellite/UAV samples, cache per resolution

Use all available cached S2 annual bands and all 64 embedding dimensions. Sample up to the configured maximum cells **per mosaic**, with at least 90% valid native UAV coverage. Preserve target and satellite-band missingness explicitly; model-specific feature filtering happens at fit time. Samples are reproducibly seeded. Full mosaics may overlap; sampled mosaic membership is retained.


In [ ]:
def features_on_grid(site,scale):
    g=grid_for(site,scale); cubes=[]; cols=[]
    for family in ('RAW_ANNUAL','EMB_ANNUAL'):
        with rasterio.open(FEATURES[(site,family)]) as ds:
            assert ds.count>0 and ds.crs==g['crs']
            for bi in range(1,ds.count+1):
                a=warp(rasterio.band(ds,bi),g,srcnodata=ds.nodata)
                a[a==-9999]=np.nan
                cubes.append(a);cols.append(f'{family}__{bi:02d}')
    return np.stack(cubes),cols
ALL_SAMPLES={}
for scale in SCALES_M:
    cache=SAMPLE/f'cells_{scale}m.parquet'
    if cache.is_file() and not FORCE_REBUILD:
        frame=pd.read_parquet(cache);ALL_SAMPLES[scale]=frame
        print('SAMPLE CACHE HIT',scale,'m',len(frame),'rows',flush=True);continue
    rows=[]
    for site,names in SITE_MOSAICS.items():
        g=grid_for(site,scale);cube,cols=features_on_grid(site,scale)
        for name in names:
            with rasterio.open(AGG_LOOKUP[(name,scale)]) as ds:
                y1,y2,valid=ds.read()
                assert ds.crs==g['crs'] and ds.transform.almost_equals(g['transform'])
            ok=(valid>=MIN_UAV_COVERAGE)&np.isfinite(y1)&np.isfinite(y2)&(y1>=0)&(y1<=1)&(y2>=0)&(y2<=1)
            # Drop missing feature cells only for the common complete-case benchmark
            ok &= np.all(np.isfinite(cube),axis=0)
            rr,cc=np.where(ok)
            rng=np.random.default_rng(SEED+sum(map(ord,name))+scale)
            take=rng.choice(len(rr),size=min(len(rr),MAX_SAMPLES_PER_MOSAIC),replace=False)
            rr=rr[take];cc=cc[take]
            frame=pd.DataFrame(cube[:,rr,cc].T,columns=cols)
            frame.insert(0,'mosaic',name);frame.insert(0,'site',site)
            frame.insert(2,'row',rr);frame.insert(3,'col',cc)
            frame['mean_probability']=y1[rr,cc];frame['cover_fraction_p05']=y2[rr,cc]
            rows.append(frame)
            print('SAMPLED',scale,'m',site,len(frame),'cells',flush=True)
        del cube;gc.collect()
    frame=pd.concat(rows,ignore_index=True)
    assert all((frame.site==s).sum()>0 for s in SITE_MOSAICS)
    frame.to_parquet(cache,index=False);ALL_SAMPLES[scale]=frame
    print('SAVED',cache,len(frame),'rows',flush=True)


## 04 — Multi-model leave-one-site-out benchmark (resumable, live progress)

Four regressors: regularized linear **Ridge**, **Random Forest**, **Extra Trees**, and **Histogram Gradient Boosting**. Fit each without using the held-out forest. Compare three predictor sets and both continuous targets. A **training-mean baseline** is included. Save *each completed fold* as a separate CSV so a Colab interruption does not discard earlier work. Fits run on CPU; GPU is not needed.


In [ ]:
def estimator(name):
    if name=='RIDGE':return make_pipeline(StandardScaler(),Ridge(alpha=20.))
    if name=='RF':return RandomForestRegressor(n_estimators=150,max_features=.65,
        min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED)
    if name=='EXTRA_TREES':return ExtraTreesRegressor(n_estimators=150,max_features=.65,
        min_samples_leaf=5,n_jobs=N_JOBS,random_state=SEED)
    if name=='HIST_GB':return HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
        max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
        early_stopping=False,random_state=SEED)
    raise ValueError(name)
def metrics(y,p):
    return dict(R2=r2_score(y,p),MAE_pp=100*mean_absolute_error(y,p),
                RMSE_pp=100*np.sqrt(mean_squared_error(y,p)),bias_pp=100*np.mean(p-y))
results=[];commenced=time.perf_counter()
for scale in SCALES_M:
    frame=ALL_SAMPLES[scale]
    for feature_set in FEATURE_SETS:
        fams=FAMILY_NAMES[feature_set]
        cols=[c for c in frame if any(c.startswith(f+'__') for f in fams)]
        for target in TARGETS:
            for heldout in SITE_MOSAICS:
                train=frame[frame.site!=heldout];test=frame[frame.site==heldout]
                assert len(train)>100 and len(test)>10
                Xtr=train[cols].to_numpy('float32');Xte=test[cols].to_numpy('float32')
                ytr=train[target].to_numpy('float32');yte=test[target].to_numpy('float32')
                for model_name in MODELS+['TRAIN_MEAN']:
                    key=f'{scale}m_{feature_set}_{target}_{heldout}_{model_name}'
                    dest=FOLDS/(key+'.csv')
                    if dest.is_file() and not FORCE_REBUILD:
                        fold=pd.read_csv(dest)
                        print('CACHE HIT',key,flush=True)
                    else:
                        t=time.perf_counter()
                        print('START',key,'train',len(ytr),'test',len(yte),flush=True)
                        if model_name=='TRAIN_MEAN':p=np.repeat(ytr.mean(),len(yte))
                        else:
                            m=estimator(model_name);m.fit(Xtr,ytr)
                            p=np.clip(m.predict(Xte),0,1)
                        fold=test[['site','mosaic','row','col']].copy()
                        fold['scale_m']=scale;fold['feature_set']=feature_set
                        fold['target']=target;fold['model']=model_name
                        fold['observed']=yte;fold['predicted']=p
                        fold.to_csv(dest,index=False)
                        met=metrics(yte,p)
                        print('DONE',key,'MAE',round(met['MAE_pp'],2),'pp',
                              'R2',round(met['R2'],3),'seconds',round(time.perf_counter()-t,1),flush=True)
                    assert len(fold)==len(yte)
                    scores=metrics(fold.observed.to_numpy(),fold.predicted.to_numpy())
                    results.append(dict(scale_m=scale,feature_set=feature_set,target=target,
                        heldout_site=heldout,model=model_name,n_test=len(fold),**scores))
    summary=pd.DataFrame(results)
    summary.to_csv(OUT/'04_LOSO_metrics_progress.csv',index=False)
    print('FINISHED scale',scale,'m; elapsed',round((time.perf_counter()-commenced)/60,1),'minutes',flush=True)
summary=pd.DataFrame(results)
summary.to_csv(OUT/'04_LOSO_metrics_full.csv',index=False)
print('=== LOSO METRICS BY SITE ===')
display(summary.round(3))
print('=== MEAN ACROSS HELD-OUT SITES (EQUAL SITE WEIGHTS) ===')
ranked=summary.groupby(['target','scale_m','feature_set','model'],as_index=False).agg(
    MAE_pp=('MAE_pp','mean'),RMSE_pp=('RMSE_pp','mean'),R2=('R2','mean'),
    bias_pp=('bias_pp','mean'))
display(ranked.sort_values(['target','MAE_pp']).round(3))
ranked.to_csv(OUT/'04_LOSO_ranked_equal_site.csv',index=False)


## 05 — Figures, diagnostic scatterplots and best-model recommendations

Show mean LOSO MAE versus scale and predictor set, and four held-out-site scatterplots for the lowest-MAE configuration separately for each UAV target. Save figures and per-site predictions; report whether satellite models beat the training-mean baseline. **Do not interpret in-sample circle scores as validation of satellite generalization.**


In [ ]:
from IPython.display import display
assert (OUT/'04_LOSO_metrics_full.csv').exists()
summary=pd.read_csv(OUT/'04_LOSO_metrics_full.csv')
ranked=pd.read_csv(OUT/'04_LOSO_ranked_equal_site.csv')
for target in TARGETS:
    entries=ranked[(ranked.target==target)&(ranked.model!='TRAIN_MEAN')].sort_values('MAE_pp')
    winner=entries.iloc[0]
    print('\nBEST',target,'scale',winner.scale_m,'m','features',winner.feature_set,
          'model',winner.model,'mean heldout-site MAE',round(winner.MAE_pp,3),'pp',flush=True)
    baseline=ranked[(ranked.target==target)&(ranked.model=='TRAIN_MEAN')&(ranked.scale_m==winner.scale_m)]
    if len(baseline):print('Training-mean baseline MAE',round(float(baseline.MAE_pp.iloc[0]),3),'pp')
    fig,axes=plt.subplots(2,2,figsize=(11,9),constrained_layout=True)
    for ax,site in zip(axes.flat,SITE_MOSAICS):
        path=FOLDS/f'{int(winner.scale_m)}m_{winner.feature_set}_{target}_{site}_{winner.model}.csv'
        df=pd.read_csv(path)
        ax.scatter(100*df.observed,100*df.predicted,s=4,alpha=.15,rasterized=True)
        ax.plot([0,100],[0,100],'--',color='grey',linewidth=1)
        metric=metrics(df.observed.to_numpy(),df.predicted.to_numpy())
        ax.set(xlim=(0,100),ylim=(0,100),title=f'{site} | MAE {metric["MAE_pp"]:.2f} pp | R² {metric["R2"]:.2f}',
               xlabel='UAV observed target (%)',ylabel='Satellite predicted target (%)')
    fig.suptitle(f'LOSO satellite: {target} | {winner.feature_set}, {int(winner.scale_m)} m, {winner.model}')
    if SAVE_FIGURES:fig.savefig(FIG/f'best_LOSO_scatter_{target}.png',dpi=175)
    plt.show();plt.close(fig)
    subset=ranked[(ranked.target==target)&(ranked.model!='TRAIN_MEAN')]
    fig,ax=plt.subplots(figsize=(10,5))
    for (features,model),g in subset.groupby(['feature_set','model']):
        ax.plot(g.scale_m,g.MAE_pp,'-o',markersize=3,label=f'{features} / {model}')
    ax.set(xlabel='Satellite grid resolution (m)',ylabel='LOSO mean MAE (percentage points)',
           title=f'{target}: out-of-site performance by resolution and model')
    ax.grid(alpha=.2);ax.legend(fontsize=7,ncol=3)
    if SAVE_FIGURES:fig.savefig(FIG/f'LOSO_MAE_scales_{target}.png',dpi=175,bbox_inches='tight')
    plt.show();plt.close(fig)
print('All tables:',OUT)
print('Figures:',FIG)


## 06 — High-cover sensitivity and global LOSO skill (no aggregation rerun)

Targeted **histogram gradient boosting** with unweighted and high-cover-weighted training, on cached 75/100/125 m sample tables. All four forests are held out in turn. This block is **standalone after Sections 00–01** (it loads cached Parquet data and does not need Section 04 to be run again). Each fit is checkpointed to Drive; rerunning skips completed folds.

Report pooled LOSO R², site-equal macro MAE, a *site-balanced* pooled R², **within-site-centred R²**, MAE skill against the corresponding training-site mean baseline, and high-cover (observed >40%) bias/recall. Figures include observed-versus-predicted scatter by site, MOS tail calibration, and overall skill comparison. **Research caveat:** model ranking on these four LOSO sites is exploratory; the same held-out sites should not later be presented as an untouched final test set.

In [ ]:
# BLOCK 06 — uses existing cached samples and archived LOSO fold predictions; no UAV reprocessing
from pathlib import Path
from IPython.display import display
import time, gc
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
assert B06_RUN
B06_DIR=OUT/'06_highcover_LOSO';B06_FOLDS=B06_DIR/'folds';B06_FIG=B06_DIR/'figures'
for d in (B06_DIR,B06_FOLDS,B06_FIG):d.mkdir(parents=True,exist_ok=True)
def b06_metrics(df):
    y=df.observed.to_numpy(float);p=df.predicted.to_numpy(float)
    out=dict(n=len(df),R2=r2_score(y,p),MAE_pp=100*np.mean(np.abs(p-y)),
             RMSE_pp=100*np.sqrt(np.mean((p-y)**2)),bias_pp=100*np.mean(p-y))
    high=y>=B06_HIGH_THRESHOLD
    out['n_high']=int(high.sum())
    out['high_bias_pp']=100*np.mean(p[high]-y[high]) if high.any() else np.nan
    out['high_MAE_pp']=100*np.mean(np.abs(p[high]-y[high])) if high.any() else np.nan
    out['high_recall']=float(np.mean(p[high]>=B06_HIGH_THRESHOLD)) if high.any() else np.nan
    return out
def b06_global(df, baseline=None):
    # Equal-site pooled scores: each site contributes total weight 1/number_of_sites.
    sz=df.groupby('site').size();weight=df.site.map(lambda s:1/sz[s]).to_numpy(float)
    weight/=weight.sum();y=df.observed.to_numpy(float);p=df.predicted.to_numpy(float)
    mse=float(np.sum(weight*(p-y)**2));mae=float(np.sum(weight*np.abs(p-y)))
    mu=float(np.sum(weight*y));sst=float(np.sum(weight*(y-mu)**2))
    # Center *both* series on the observed site mean to suppress between-site variation.
    means=df.groupby('site').observed.mean()
    yc=y-df.site.map(means).to_numpy(float)
    pc=p-df.site.map(means).to_numpy(float)
    within_sst=float(np.sum(weight*yc**2))
    val=dict(pooled_R2=r2_score(y,p),balanced_pooled_R2=1-mse/sst if sst>0 else np.nan,
             within_site_R2=1-mse/within_sst if within_sst>0 else np.nan,
             macro_MAE_pp=100*mae,macro_RMSE_pp=100*np.sqrt(mse),
             macro_bias_pp=100*np.sum(weight*(p-y)))
    vals=[b06_metrics(g) for _,g in df.groupby('site')]
    val['macro_high_bias_pp']=float(np.nanmean([v['high_bias_pp'] for v in vals])) if any(v['n_high'] for v in vals) else np.nan
    val['macro_high_recall']=float(np.nanmean([v['high_recall'] for v in vals])) if any(v['n_high'] for v in vals) else np.nan
    if baseline is not None:
        base=b06_global(baseline)
        val['MAE_skill_vs_train_mean']=1-val['macro_MAE_pp']/base['macro_MAE_pp']
    return val
b06_records=[];b06_start=time.perf_counter()
for scale in B06_SCALES_M:
    sample_path=SAMPLE/f'cells_{scale}m.parquet'
    assert sample_path.is_file(),f'Run Section 03 once to produce {sample_path}'
    frame=pd.read_parquet(sample_path)
    print(f'\\nBLOCK 06 | {scale} m | {len(frame):,} cached cells',flush=True)
    for feature_set in B06_FEATURE_SETS:
        cols=[c for c in frame if any(c.startswith(f+'__') for f in FAMILY_NAMES[feature_set])]
        for target in B06_TARGETS:
            for holdout in SITE_MOSAICS:
                tr=frame[frame.site!=holdout];te=frame[frame.site==holdout]
                Xtr=tr[cols].to_numpy('float32');Xte=te[cols].to_numpy('float32')
                ytr=tr[target].to_numpy('float32');yte=te[target].to_numpy('float32')
                for strength in B06_WEIGHT_STRENGTHS:
                    tag=f'{scale}m_{feature_set}_{target}_{holdout}_w{strength}'
                    dest=B06_FOLDS/f'{tag}.csv'
                    if dest.exists() and not B06_REBUILD:
                        fold=pd.read_csv(dest)
                        print('CACHE',tag,flush=True)
                    else:
                        t0=time.perf_counter()
                        # Smooth upweighting begins at 20%, saturates at 60%;
                        # e.g. strength=5 yields weights from 1 to 6.
                        weights=1+strength*np.clip((ytr-.20)/.40,0,1)
                        model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                            max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                            early_stopping=False,random_state=SEED)
                        model.fit(Xtr,ytr,sample_weight=weights)
                        pred=np.clip(model.predict(Xte),0,1)
                        fold=te[['site','mosaic','row','col']].copy()
                        fold['observed']=yte;fold['predicted']=pred
                        fold.to_csv(dest,index=False)
                        print(f'DONE {tag} | MAE={100*np.mean(abs(pred-yte)):.2f} pp | R²={r2_score(yte,pred):.3f} | {time.perf_counter()-t0:.1f}s',flush=True)
                    b06_records.append(dict(scale_m=scale,feature_set=feature_set,target=target,
                                            holdout=holdout,strength=strength,**b06_metrics(fold)))
    del frame;gc.collect()
pd.DataFrame(b06_records).to_csv(B06_DIR/'06_per_site_metrics.csv',index=False)
global_records=[]
for scale in B06_SCALES_M:
 for feature_set in B06_FEATURE_SETS:
  for target in B06_TARGETS:
   baseline_parts=[]
   for site in SITE_MOSAICS:
    bpath=FOLDS/f'{scale}m_{feature_set}_{target}_{site}_TRAIN_MEAN.csv'
    if bpath.exists():baseline_parts.append(pd.read_csv(bpath)[['site','observed','predicted']])
    else:
     f=pd.read_parquet(SAMPLE/f'cells_{scale}m.parquet')
     tr=f[f.site!=site];te=f[f.site==site]
     baseline_parts.append(pd.DataFrame(dict(site=site,observed=te[target].to_numpy(),
                                          predicted=np.repeat(tr[target].mean(),len(te)))))
   baseline=pd.concat(baseline_parts,ignore_index=True)
   for strength in B06_WEIGHT_STRENGTHS:
    ds=[pd.read_csv(B06_FOLDS/f'{scale}m_{feature_set}_{target}_{site}_w{strength}.csv') for site in SITE_MOSAICS]
    df=pd.concat(ds,ignore_index=True)
    global_records.append(dict(scale_m=scale,feature_set=feature_set,target=target,strength=strength,
                               **b06_global(df,baseline)))
global_table=pd.DataFrame(global_records).sort_values(['target','macro_MAE_pp'])
global_table.to_csv(B06_DIR/'06_global_LOSO_metrics.csv',index=False)
display(global_table.round(3))
for target in B06_TARGETS:
 group=global_table[global_table.target==target]
 winner=group.sort_values('macro_MAE_pp').iloc[0]
 scale=int(winner.scale_m);family=winner.feature_set;strength=int(winner.strength)
 ds=[pd.read_csv(B06_FOLDS/f'{scale}m_{family}_{target}_{site}_w{strength}.csv') for site in SITE_MOSAICS]
 best=pd.concat(ds,ignore_index=True)
 fig,axes=plt.subplots(2,2,figsize=(10,9),constrained_layout=True)
 for ax,(site,g) in zip(axes.flat,best.groupby('site',sort=False)):
    ax.hexbin(g.observed*100,g.predicted*100,gridsize=38,mincnt=1,bins='log',cmap='viridis')
    ax.plot([0,100],[0,100],'--',color='gray',lw=1)
    mm=b06_metrics(g);ax.set(xlim=(0,100),ylim=(0,100),xlabel='UAV target (%)',
        ylabel='Satellite predicted (%)',title=f'{site}: R² {mm["R2"]:.2f}, MAE {mm["MAE_pp"]:.1f} pp')
 fig.suptitle(f'Weighted boosting LOSO | {target} | {scale}m {family} weight={strength}')
 fig.savefig(B06_FIG/f'06_best_LOSO_scatter_{target}.png',dpi=175)
 plt.show();plt.close(fig)
 # MOS calibration: observed abundance bins; inspect high-cover saturation explicitly.
 mos=best[best.site=='MOS'].copy()
 mos['bin']=pd.cut(mos.observed,bins=[0,.1,.2,.3,.4,.5,.6,.7,.8,1.0001],include_lowest=True)
 cal=mos.groupby('bin',observed=True).agg(n=('observed','size'),
        observed=('observed','mean'),predicted=('predicted','mean')).reset_index()
 cal.to_csv(B06_DIR/f'06_MOS_calibration_{target}.csv',index=False)
 fig,ax=plt.subplots(figsize=(7,5))
 ax.plot([0,100],[0,100],'--',color='gray',lw=1)
 ax.plot(cal.observed*100,cal.predicted*100,'o-',label='MOS binned mean')
 for _,rr in cal.iterrows():ax.annotate(str(int(rr['n'])),(100*rr.observed,100*rr.predicted),xytext=(4,4),textcoords='offset points',fontsize=7)
 ax.set(xlim=(0,100),ylim=(0,100),xlabel='Observed MOS fraction (%)',ylabel='Predicted MOS fraction (%)',
        title=f'MOS tail calibration: {target} | {scale}m {family} w={strength}')
 ax.legend();ax.grid(alpha=.2);fig.tight_layout()
 fig.savefig(B06_FIG/f'06_MOS_tail_{target}.png',dpi=175)
 plt.show();plt.close(fig)
 print(f'BEST {target}: {scale}m {family} weight={strength} | macro MAE={winner.macro_MAE_pp:.2f}pp | pooled R²={winner.pooled_R2:.3f} | within-site R²={winner.within_site_R2:.3f}',flush=True)
print('BLOCK 06 saved:',B06_DIR,'| elapsed min',round((time.perf_counter()-b06_start)/60,1),flush=True)
